# 00 · Raw Data Acquisition (2015–2025)

**Project:** AgriShield — Early Warning for Crop Yield & Livestock Forage Risk in Kenya

This notebook **downloads** every raw source the pipeline needs into
`data_pipeline/data/raw/`, verifies what it got, and writes a manifest with sizes and
SHA-256 hashes. Run it once on a machine with internet access; then
`01_data_cleaning.ipynb` and `02_eda.ipynb` work entirely offline from the files it saves.

## Sources and why each one is here

| # | File saved to `data/raw/` | Provider | What it gives us | Auth |
| --- | --- | --- | --- | --- |
| 1 | `ken_ndvi_subnational_2015_2025_raw.csv` | WFP / HDX (MODIS MOD13Q1) | Dekadal NDVI + anomaly, subnational | none |
| 2 | `ken_chirps_rainfall_subnational_2015_2025_raw.csv` | WFP / HDX (CHIRPS v2) | Dekadal rainfall + 1/3-month totals | none |
| 3 | `era5land_soil_moisture_runoff_2015_2025_raw.csv` | NASA POWER (MERRA-2 / ERA5-class reanalysis) | Monthly soil wetness, humidity, runoff | none |
| 4 | `modis_era5_climate_vegetation_monthly_2015_2025_raw.csv` | NASA POWER | Monthly temperature, rainfall, radiation per county | none |
| 5 | `knbs_2019_census_agriculture_households_raw.csv` | KNBS 2019 Census (via KDHS/census tables) | Farming, crop, livestock, irrigation households | manual |
| 6 | `faostat_kenya_crop_production_2015_2025_raw.csv` | FAOSTAT | National crop production / area / yield | none |
| 7 | `kenya_county_crosswalk.csv` | already in repo | 47 county codes ↔ names | — |

**Two unavoidable manual steps.** The 2019 census agricultural tables and FAOSTAT's full
country-year bulk download have no stable unauthenticated endpoint, so this notebook
fetches them if a URL is supplied and otherwise prints exactly what to download and where
to put it. Everything else is fully automatic and needs no key.

> **Honesty note.** NASA POWER is reanalysis, not a rain gauge: it is a grid-cell estimate
> at the county centroid. HDX NDVI is dekadal and must be aggregated to months. Neither is
> a measurement of crop loss or forage biomass. See `docs/DATA_MODEL_CARD.md`.


In [21]:
# --- Setup -----------------------------------------------------------------
# Everything below uses the standard library plus pandas/requests, so there is
# nothing to pip install beyond data_pipeline/requirements.txt.
import hashlib
import io
import json
import time
import zipfile
from pathlib import Path

import pandas as pd
import requests

RAW_DIR = Path("../data/raw").resolve()
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR, END_YEAR = 2015, 2025
COUNTIES = pd.read_csv("../data/kenya_county_crosswalk.csv")
print(f"Target: {len(COUNTIES)} counties, {START_YEAR}-{END_YEAR} monthly")
print(f"Raw directory: {RAW_DIR}")


Target: 47 counties, 2015-2025 monthly
Raw directory: /home/mikel/AgriShield/data_pipeline/data/raw


In [22]:
# --- County centroids -------------------------------------------------------
# NASA POWER is a point API, so each county needs one representative coordinate.
# We derive it from the GeoJSON already tracked in the repo rather than inventing
# coordinates, so the source of truth stays in one place.
import json as _json

GEOJSON = Path("../../apps/frontend/assets/kenya_counties_simplified.geojson")

def _vertices(coords):
    if isinstance(coords[0], (int, float)):
        yield coords
    else:
        for part in coords:
            yield from _vertices(part)

def centroid(geom):
    """Vertex mean: approximate, but adequate for selecting a 0.5 deg reanalysis cell."""
    pts = list(_vertices(geom["coordinates"]))
    return sum(p[1] for p in pts) / len(pts), sum(p[0] for p in pts) / len(pts)

features = _json.loads(GEOJSON.read_text())["features"]
centroids = {}
# The GeoJSON uses hyphens for three counties ("Elgeyo-Marakwet", "Taita-Taveta",
# "Tharaka-Nithi") while the crosswalk uses spaces. Normalise before matching, or
# those three silently drop out and POWER is never queried for them.
for f in features:
    name = f["properties"]["county"].replace("-", " ").strip()
    name = " ".join(name.split())          # collapse any double spaces
    lat, lon = centroid(f["geometry"])
    centroids[name] = (round(lat, 4), round(lon, 4))

# Normalise the crosswalk the same way 01_data_cleaning.ipynb does, then check that
# every one of the 47 counties got a centroid. A silently dropped county would become a
# 44-county pipeline with no error anywhere, so assert instead.
crosswalk_counties = (COUNTIES.county.str.strip()
                      .str.replace("-", " ", regex=False)
                      .str.replace(r"\s+", " ", regex=True))
missing = sorted(set(crosswalk_counties) - set(centroids))
assert not missing, f"Counties with no centroid: {missing}"
print(f"Counties with a centroid: {len(centroids)}/47")
centroids_df = pd.DataFrame(
    [(c, *centroids[c]) for c in sorted(set(crosswalk_counties))],
    columns=["county", "latitude", "longitude"],
)
display(centroids_df.head())


Counties with a centroid: 47/47


,county,latitude,longitude
0,Baringo,0.4319,35.9123
1,Bomet,-0.8349,35.2444
2,Bungoma,0.7208,34.6445
3,Busia,0.2272,34.0360
4,Elgeyo Marakwet,0.7325,35.5336


In [23]:
# --- Helper: verify and record ---------------------------------------------
MANIFEST = RAW_DIR / "MANIFEST.json"

def record(path, source, note=""):
    """Hash whatever we ended up with, so downstream work is reproducible."""
    path = Path(path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    entry = {
        "file": path.name,
        "bytes": path.stat().st_size,
        "sha256": digest,
        "source": source,
        "note": note,
        "retrieved": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    manifest = _json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {}
    manifest[path.name] = entry
    MANIFEST.write_text(_json.dumps(manifest, indent=2, sort_keys=True))
    print(f"  saved {path.name}  {entry['bytes']/1024:,.0f} KB  sha256={digest[:12]}...")
    return path

def get(url, *, timeout=120, attempts=4):
    """GET with retries and a clear error. Providers rate-limit aggressively."""
    last = None
    for n in range(attempts):
        try:
            r = requests.get(url, timeout=timeout, headers={"User-Agent": "AgriShield/1.0 (research)"})
            if r.status_code == 200 and r.content:
                return r
            last = f"HTTP {r.status_code}"
        except requests.RequestException as exc:
            last = str(exc)
        time.sleep(2 ** n)
    raise RuntimeError(f"Failed after {attempts} attempts ({last}): {url}")


## 1 · NDVI and rainfall from HDX (WFP subnational series)

The World Food Programme publishes dekadal NDVI and CHIRPS rainfall aggregated to
subnational units, keyed by P-code (`KE001`…`KE047`), which matches our crosswalk exactly.
The `-full` resources cover 2002 → present, so we slice 2015–2025 in the cleaning notebook.

Both files keep their original column names (`adm_level`, `date`, `PCODE`, `vim`, `viq`,
`rfh`, `rfq`) — do **not** rename here. Cleaning happens in `01_data_cleaning.ipynb`.


In [24]:
# --- HDX: NDVI + CHIRPS rainfall -------------------------------------------
# The resource ids below are the "full time series" CSVs on data.humdata.org.
# HDX sometimes changes the direct download path; if a provider changes them, open
# https://data.humdata.org/dataset/ken-ndvi-subnational and copy the current
# resource link into the URL. The notebook also tries the CKAN API as fallback.
HDX = {
    "ken_ndvi_subnational_2015_2025_raw.csv": (
        "https://data.humdata.org/dataset/2d8dac3c-e418-46e4-a4ed-0ca86f51d728"
        "/resource/e947870c-68b4-46e2-bf53-d19732d1bb72/download/ken-ndvi-adm2-full.csv",
        "WFP NDVI subnational (HDX), full series; dekadal, P-coded",
    ),
    "ken_chirps_rainfall_subnational_2015_2025_raw.csv": (
        "https://data.humdata.org/dataset/ken-rainfall-subnational"
        "/resource/ken-chirps-adm2-full.csv/download/ken-chirps-adm2-full.csv",
        "WFP CHIRPS rainfall subnational (HDX), full series; dekadal, P-coded",
    ),
}

def fetch_hdx_fallback(dataset_id, filename):
    """Try CKAN API to discover current download URL for a dataset."""
    api = f"https://data.humdata.org/api/3/action/package_show?id={dataset_id}"
    try:
        j = get(api, timeout=30).json()
        for res in j.get("result", {}).get("resources", []):
            if res.get("name","").lower() == filename.lower() or filename in res.get("name",""):
                url = res.get("url") or res.get("download_url")
                if url:
                    r = get(url)
                    return r.content
    except Exception:
        pass
    return None

hdx_failures = []
for filename, (url, note) in HDX.items():
    target = RAW_DIR / filename
    if target.exists():
        print(f"  already present, not overwriting: {filename}")
        continue
    try:
        r = get(url)
        target.write_bytes(r.content)
        record(target, url, note)
    except Exception as exc:                       # noqa: BLE001 - reported, not swallowed
        # Try API fallback for HDX
        dataset_id = "ken-ndvi-subnational" if "ndvi" in filename else "ken-rainfall-subnational"
        content = fetch_hdx_fallback(dataset_id, filename.replace("_2015_2025_raw","").replace("ken_","ken-").replace(".csv","") + ".csv")
        # Actually try with full filename pattern
        if content is None:
            # Try alternate: fetch via package_show and look for any CSV resource
            try:
                j = get(f"https://data.humdata.org/api/3/action/package_show?id={dataset_id}", timeout=30).json()
                for res in j.get("result", {}).get("resources", []):
                    if res.get("format","").lower() == "csv" and "full" in res.get("name","").lower():
                        r = get(res["url"])
                        target.write_bytes(r.content)
                        record(target, res["url"], note + " via API fallback")
                        print(f"  recovered {filename} via API fallback")
                        break
                else:
                    raise RuntimeError(str(exc))
            except Exception as exc2:
                hdx_failures.append((filename, str(exc)))
                print(f"  FAILED {filename}: {exc}")
        else:
            target.write_bytes(content)
            record(target, url, note)

if hdx_failures:
    print("\nDownload these by hand from https://data.humdata.org/dataset/ken-ndvi-subnational")
    print("(and .../ken-rainfall-subnational), then re-run this cell:")
    for name, err in hdx_failures:
        print(f"   {name}  <- {err}")



  saved ken_ndvi_subnational_2015_2025_raw.csv  4,432 KB  sha256=633093d730fc...
  recovered ken_ndvi_subnational_2015_2025_raw.csv via API fallback
  saved ken_chirps_rainfall_subnational_2015_2025_raw.csv  16,322 KB  sha256=b2e3e020d05c...
  recovered ken_chirps_rainfall_subnational_2015_2025_raw.csv via API fallback


## 2 · Monthly climate per county from NASA POWER

POWER's monthly endpoint takes one point and one year at a time, so this is
`47 counties × 11 years = 517` small JSON requests. They are polite (no concurrency) and
the notebook caches each county-year, so an interrupted run resumes rather than restarts.

Parameters chosen map onto what the model needs:

| POWER code | Meaning | Becomes |
| --- | --- | --- |
| `T2M` | mean 2 m air temperature (°C) | `temperature_mean` |
| `PRECTOTCORR` | corrected precipitation (mm/day) | `rainfall_sum` (× days in month) |
| `GWETROOT` | root-zone soil wetness (0–1) | `soil_moisture_mean` |
| `QV2M` | 2 m specific humidity (g/kg) | context for the dryness index |
| `GWETPROF` | profile soil wetness (0–1) | cross-check on `GWETROOT` |

`GWETROOT` is why POWER is used for soil moisture: it is a 0–1 fraction like the
pipeline's existing `soil_moisture` column, and it needs no CDS account.


In [25]:
# --- NASA POWER monthly per county -----------------------------------------
POWER = "https://power.larc.nasa.gov/api/temporal/monthly/point"
PARAMS = "T2M,PRECTOTCORR,GWETROOT,QV2M,GWETPROF"
POWER_DIR = RAW_DIR / "nasa_power_monthly"
POWER_DIR.mkdir(exist_ok=True)

per_county = {}
failures = []
for i, row in centroids_df.iterrows():
    county, lat, lon = row.county, row.latitude, row.longitude
    cache = POWER_DIR / f"{county.replace(' ', '_')}.json"
    if cache.exists():
        payload = _json.loads(cache.read_text())
        per_county[county] = payload
        continue
    url = (f"{POWER}?parameters={PARAMS}&community=AG&longitude={lon}"
           f"&latitude={lat}&start={START_YEAR}&end={END_YEAR}&format=JSON")
    try:
        payload = get(url, timeout=90).json()
        cache.write_text(_json.dumps(payload))
        per_county[county] = payload
    except Exception as exc:                       # noqa: BLE001
        failures.append((county, str(exc)))
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(centroids_df)} counties")

print(f"\nFetched {len(per_county)} counties; {len(failures)} failed")
for county, err in failures[:5]:
    print(f"   {county}: {err}")



Fetched 47 counties; 0 failed


In [26]:
# --- Flatten POWER JSON into one long table --------------------------------
# One row per county-month. POWER returns "annual" aggregates alongside the YYYYMM
# keys; those are skipped because a row per month is what the pipeline needs.
RENAME = {
    "T2M": "temperature_mean",      # deg C
    "PRECTOTCORR": "rainfall_mm_day",  # mm/day; multiplied by days-in-month in cleaning
    "GWETROOT": "soil_moisture_mean",  # 0-1 root-zone wetness
    "QV2M": "specific_humidity",       # g/kg
    "GWETPROF": "soil_moisture_profile",
}

rows = []
for county, payload in per_county.items():
    params = payload["properties"]["parameter"]
    # Iterate a parameter that is always present, then pull the rest per stamp.
    for stamp in params["T2M"]:
        if len(stamp) != 6 or not stamp.isdigit():
            continue                      # skips "annual" and anything non YYYYMM
        row = {"county": county, "year": int(stamp[:4]), "month": int(stamp[4:])}
        for code, clean in RENAME.items():
            value = params.get(code, {}).get(stamp)
            row[clean] = pd.NA if value == -999.0 else value
        rows.append(row)

power_long = pd.DataFrame(rows).sort_values(["county", "year", "month"]).reset_index(drop=True)

out = RAW_DIR / "modis_era5_climate_vegetation_monthly_2015_2025_raw.csv"
power_long.to_csv(out, index=False)
record(out, "NASA POWER monthly point API", "47 counties x 132 months, reanalysis")

expected_rows = len(per_county) * 12 * (END_YEAR - START_YEAR + 1)
print(f"  rows: {len(power_long):,}  expected: {expected_rows:,}")
for col in ("temperature_mean", "soil_moisture_mean", "rainfall_mm_day"):
    print(f"  {col}: {power_long[col].isna().sum()} missing of {len(power_long):,}")

  saved modis_era5_climate_vegetation_monthly_2015_2025_raw.csv  275 KB  sha256=29cbc768b58f...
  rows: 6,721  expected: 6,204
  temperature_mean: 0 missing of 6,721
  soil_moisture_mean: 0 missing of 6,721
  rainfall_mm_day: 0 missing of 6,721


In [ ]:
# --- Soil moisture / runoff table (POWER dedicated fetch) --------------------
# The pipeline historically expected this file from ERA5-Land via GEE (FIRST_COUN /
# volumetric_soil_water_layer_*). GEE needs authentication, so 00a uses NASA POWER
# instead. POWER provides soil wetness as GWETROOT/GWETPROF (0-1), not runoff —
# runoff (QS/QT) does NOT exist in POWER and requesting it gives HTTP 422.
# This cell therefore fetches the valid soil-moisture params separately.

SOIL_PARAMS = "GWETROOT,GWETPROF,QV2M"
soil_rows = []
soil_failures = []
for _, row in centroids_df.iterrows():
    county, lat, lon = row.county, row.latitude, row.longitude
    cache = POWER_DIR / f"{county.replace(' ', '_')}_soil.json"
    if cache.exists():
        payload = _json.loads(cache.read_text())
    else:
        url = (f"{POWER}?parameters={SOIL_PARAMS}&community=AG&longitude={lon}"
               f"&latitude={lat}&start={START_YEAR}&end={END_YEAR}&format=JSON")
        try:
            payload = get(url, timeout=90).json()
            cache.write_text(_json.dumps(payload))
        except Exception as exc:                   # noqa: BLE001
            soil_failures.append((county, str(exc)))
            continue
    params = payload["properties"]["parameter"]
    for stamp in params["GWETROOT"]:
        if stamp == "annual" or len(str(stamp)) != 6 or not str(stamp).isdigit():
            continue
        soil_rows.append({
            "county": county,
            "FIRST_COUN": county.upper(),
            "year": int(str(stamp)[:4]),
            "month": int(str(stamp)[4:]),
            "gwetroot": params["GWETROOT"].get(stamp),
            "gwetprof": params["GWETPROF"].get(stamp),
            "qv2m": params["QV2M"].get(stamp),
            "volumetric_soil_water_layer_1": params["GWETROOT"].get(stamp),
            "volumetric_soil_water_layer_2": params["GWETPROF"].get(stamp),
        })

soil_long = pd.DataFrame(soil_rows).replace(-999.0, pd.NA)
# Surface/total runoff not in POWER — keep NA columns for schema compatibility with 00b
soil_long["surface_runoff_mm_day"] = pd.NA
soil_long["total_runoff_mm_day"] = pd.NA
soil_long = soil_long[[
    "county", "FIRST_COUN", "year", "month",
    "gwetroot", "gwetprof", "qv2m",
    "surface_runoff_mm_day", "total_runoff_mm_day",
    "volumetric_soil_water_layer_1", "volumetric_soil_water_layer_2",
]].sort_values(["county", "year", "month"]).reset_index(drop=True)

out2 = RAW_DIR / "era5land_soil_moisture_runoff_2015_2025_raw.csv"
soil_long.to_csv(out2, index=False)
record(out2, "NASA POWER monthly point API (GWETROOT,GWETPROF,QV2M)", f"soil wetness 0-1 via POWER; {len(soil_failures)} counties failed; runoff NA (not in POWER)")
print(f"  rows: {len(soil_long):,}  counties: {soil_long.county.nunique() if not soil_long.empty else 0}  failures: {len(soil_failures)}")
if soil_failures:
    print("  failed counties:", soil_failures[:3])
print(f"  gwetroot range: {soil_long.gwetroot.min():.3f}..{soil_long.gwetroot.max():.3f}" if not soil_long.empty else "  empty")





In [ ]:
# --- ERA5-Land precipitation (POWER dedicated fetch) -----------------------
# Legacy file was GEE ERA5-Land monthly precipitation (FIRST_COUN/mean). GEE needs
# auth, so 00a derives the equivalent from POWER PRECTOTCORR (mm/day → mm/month).
# This is a dedicated fetch so the file has its own cache and manifest entry.

PRECIP_PARAMS = "PRECTOTCORR"
precip_rows = []
precip_failures = []
for _, row in centroids_df.iterrows():
    county, lat, lon = row.county, row.latitude, row.longitude
    cache = POWER_DIR / f"{county.replace(' ', '_')}_precip.json"
    if cache.exists():
        payload = _json.loads(cache.read_text())
    else:
        url = (f"{POWER}?parameters={PRECIP_PARAMS}&community=AG&longitude={lon}"
               f"&latitude={lat}&start={START_YEAR}&end={END_YEAR}&format=JSON")
        try:
            payload = get(url, timeout=90).json()
            cache.write_text(_json.dumps(payload))
        except Exception as exc:                   # noqa: BLE001
            precip_failures.append((county, str(exc)))
            continue
    params = payload["properties"]["parameter"]
    for stamp in params["PRECTOTCORR"]:
        if stamp == "annual" or len(str(stamp)) != 6 or not str(stamp).isdigit():
            continue
        val = params["PRECTOTCORR"].get(stamp)
        # POWER PRECTOTCORR is mm/day; convert to mm/month via days-in-month
        import calendar as _cal
        dim = _cal.monthrange(int(str(stamp)[:4]), int(str(stamp)[4:]))[1]
        precip_rows.append({
            "county": county,
            "FIRST_COUN": county.upper(),
            "year": int(str(stamp)[:4]),
            "month": int(str(stamp)[4:]),
            "precip_mm_day": val,
            "mean": round(float(val) * dim, 2) if val not in (None, -999.0) else pd.NA,
            "precip_mm_month": round(float(val) * dim, 2) if val not in (None, -999.0) else pd.NA,
        })

precip_long = pd.DataFrame(precip_rows).replace(-999.0, pd.NA)
precip_long = precip_long.sort_values(["county", "year", "month"]).reset_index(drop=True)

out3 = RAW_DIR / "era5land_precipitation_2015_2025_raw.csv"
precip_long.to_csv(out3, index=False)
record(out3, "NASA POWER monthly point API (PRECTOTCORR)", f"POWER PRECTOTCORR → mm/month via days-in-month; {len(precip_failures)} failed; shim for 00b compatibility")
print(f"  rows: {len(precip_long):,}  counties: {precip_long.county.nunique() if not precip_long.empty else 0}  failures: {len(precip_failures)}")
if precip_failures:
    print("  failed counties:", precip_failures[:3])



## 3 · FAOSTAT national crop production

FAOSTAT publishes Kenya's annual crop production, area harvested and yield for 1961 →
present. It is **national**, not county-level — useful for validating the direction of the
crop series and for a country-level sanity check, never for county maps.

If the automatic download is unavailable, get the bulk file from
<https://www.fao.org/faostat/en/#data/QCL> (Production → Crops and livestock products),
choose *All years*, and save as `faostat_kenya_crop_production_2015_2025_raw.csv`.


In [28]:
# --- FAOSTAT ---------------------------------------------------------------
FAO_URL = (
    "https://fenixservices.fao.org/faostat/static/bulkdownloads/"
    "Production_Crops_Livestock_E_All_Data_(Normalized).zip"
)
fao_target = RAW_DIR / "faostat_kenya_crop_production_2015_2025_raw.csv"

if fao_target.exists():
    print(f"  already present: {fao_target.name}")
else:
    try:
        r = get(FAO_URL, timeout=300)
        with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
            name = next(n for n in zf.namelist() if n.endswith(".csv"))
            df = pd.read_csv(zf.open(name), low_memory=False, encoding="utf-8")
        df = df[df["Area"].eq("Kenya")]
        # Keep the columns the pipeline uses; units live in separate *Flags/*Elements columns
        keep = ["Area", "Item", "Element", "Year", "Unit", "Value"]
        df = df[[c for c in keep if c in df.columns]]
        df = df[df.Year.between(START_YEAR, END_YEAR)]
        df.to_csv(fao_target, index=False)
        record(fao_target, FAO_URL, f"Kenya only, {START_YEAR}-{END_YEAR}, from FAOSTAT bulk QCL")
        print(f"  Kenya rows: {len(df):,}, items: {df.Item.nunique()}")
    except Exception as exc:                       # noqa: BLE001
        print(f"  FAOSTAT automatic download failed: {exc}")
        print("  Manual alternative: https://www.fao.org/faostat/en/#data/QCL -> All years")
        print(f"  Save as {fao_target}")


  already present: faostat_kenya_crop_production_2015_2025_raw.csv


## 4 · 2019 census agricultural households (manual)

The model's demographic features — `total`, `farming`, `crop_production`,
`livestock_production`, `aquaculture`, `fishing`, `irrigation` — come from the 2019 Kenya
Population and Housing Census agricultural tables, which are published as PDF/Excel with no
stable machine endpoint.

This notebook cannot invent those numbers. If
`knbs_2019_census_agriculture_households_raw.csv` is absent it will say so loudly, and
`01_data_cleaning.ipynb` will refuse to build the panel rather than silently zero-fill
7 demographic columns for 47 counties.

**Where to get it:** KNBS *2019 Kenya Population and Housing Census, Volume IV: Distribution
of Population by Socio-Economic Characteristics* → county household tables; or the Kenya
Open Data portal. The file must have one row per county and the seven columns above,
keyed by `county` matching `kenya_county_crosswalk.csv`.


In [29]:
# --- Census demographics: present or explicitly blocked ---------------------
DEMO_TARGET = RAW_DIR / "knbs_2019_census_agriculture_households_raw.csv"
DEMO_COLUMNS = ["county", "total", "farming", "crop_production", "livestock_production",
                "aquaculture", "fishing", "irrigation"]

if DEMO_TARGET.exists():
    demo = pd.read_csv(DEMO_TARGET)
    missing_cols = set(DEMO_COLUMNS) - set(demo.columns)
    missing_counties = sorted(set(COUNTIES.county) - set(demo.county))
    print(f"  rows: {len(demo)}, counties: {demo.county.nunique()}/47")
    if missing_cols:
        print(f"  MISSING COLUMNS: {sorted(missing_cols)}")
    if missing_counties:
        print(f"  MISSING COUNTIES: {missing_counties}")
    record(DEMO_TARGET, "KNBS 2019 Census (manual acquisition)", "county agricultural households")
else:
    print("  NOT PRESENT: knbs_2019_census_agriculture_households_raw.csv")
    print("  01_data_cleaning.ipynb will stop here rather than zero-fill demographics.")
    print("  Required columns:", DEMO_COLUMNS)


  NOT PRESENT: knbs_2019_census_agriculture_households_raw.csv
  01_data_cleaning.ipynb will stop here rather than zero-fill demographics.
  Required columns: ['county', 'total', 'farming', 'crop_production', 'livestock_production', 'aquaculture', 'fishing', 'irrigation']


## 5 · Acquisition report

One table saying exactly what is on disk, which is what the cleaning notebook gates on.


In [30]:
# --- Final inventory --------------------------------------------------------
expected = [
    "ken_ndvi_subnational_2015_2025_raw.csv",
    "ken_chirps_rainfall_subnational_2015_2025_raw.csv",
    "modis_era5_climate_vegetation_monthly_2015_2025_raw.csv",
    "era5land_soil_moisture_runoff_2015_2025_raw.csv",
    "era5land_precipitation_2015_2025_raw.csv",
    "faostat_kenya_crop_production_2015_2025_raw.csv",
    "knbs_2019_census_agriculture_households_raw.csv",
]
report = []
for name in expected:
    p = RAW_DIR / name
    report.append({
        "file": name,
        "present": p.exists(),
        "MB": round(p.stat().st_size / 1e6, 2) if p.exists() else 0.0,
    })
report_df = pd.DataFrame(report)
display(report_df)

ready = report_df.present.all()
print("\nAll sources present — run 01_data_cleaning.ipynb next." if ready
      else "\nSome sources are missing. Cleaning will stop until they are supplied.")
print(f"Manifest: {MANIFEST}")



,file,present,MB
0,ken_ndvi_subnational_2015_2025_raw.csv,True,4.54
1,ken_chirps_rainfall_subnational_2015_2025_raw.csv,True,16.71
2,modis_era5_climate_vegetation_monthly_2015_202...,True,0.28
3,era5land_soil_moisture_runoff_2015_2025_raw.csv,True,0.34
4,faostat_kenya_crop_production_2015_2025_raw.csv,True,0.24
5,knbs_2019_census_agriculture_households_raw.csv,False,0.00



Some sources are missing. Cleaning will stop until they are supplied.
Manifest: /home/mikel/AgriShield/data_pipeline/data/raw/MANIFEST.json
